# signature_discount

Genome-wide instability signatures (MSI, CIN, aneuploidy) describe a whole
**cell line**, not any single gene. They tell the scorer how much to **trust**
a gene-level alteration found in that line:

- A point mutation in a mismatch-repair-deficient (MSI-high) line is far more
  likely to be a random passenger, because such lines are hypermutators.
- A fusion in a chromosomally unstable (high-CIN) line is more likely to be a
  passenger rearrangement.

This module converts the raw, differently-scaled instability metrics into two
clean per-cell-line **discount multipliers** in `(0, 1]`:

- `m_mut` -> dims the mutation evidence for that line (driven by MSI)
- `m_fus` -> dims the fusion evidence for that line (driven by CIN + aneuploidy)

These are applied downstream, immediately **before** the noisy-OR that combines
mutation + fusion into a structural score:

```
s_mut'   = m_mut * (burden component of s_mut)   # see note below
s_fus'   = m_fus * s_fus
s_struct = 1 - (1 - s_mut') * (1 - s_fus')
```

**Shape** - floored reversed logistic (same Hill family as the evidence curves, inverted):

```
m(p) = m_min + (1 - m_min) / (1 + exp(k * (p - p0)))
```

- `p` : cohort instability percentile for this line, in `(0, 1]`
- `p0` : percentile where the discount is half-applied
- `k` : steepness (MSI is quasi-binary -> steep; CIN is continuous -> gentle)
- `m_min` : floor. Hard lower bound > 0 so a real driver in a hypermutator
  (e.g. KRAS G13D in HCT116) is never zeroed out.

**Does / does not**
- **Does**: percentile-rank instability across the cohort, apply the discount,
  quarantine missing-data lines (`m = 1.0`), emit an auditable table.
- **Does not**: decide *which part* of the mutation score to multiply. Biology
  says MSI inflates mutation *quantity*, not *quality*, so the scorer applies
  `m_mut` to the burden component only and leaves the oncogene/pathogenicity
  gate intact. That targeting lives in the scorer, not here.

**Excluded by design**
- `lohfraction` - LOH is the TSG second-hit *mechanism* (Knudson), not noise -
  discounting on it would penalise the exact signal we hunt for.
- `wgd`, `ploidy` - context / VAF-correction factors, not trust discounts.

Columns: `model_id`, `msi_pctl`, `instab_pctl`, `m_mut`, `m_fus`,
`mut_discount_applied`, `fus_discount_applied`

The two audit flags are independent: MSI has wider coverage than the structural
(CIN/aneuploidy) block, so a line can have its mutation discount applied while
its fusion discount defaults to 1.0. Never penalise a line for an absent metric.

In [ ]:
import os
import sys

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
# src/Track - C/track-C-scoring/ -> up two levels to src/, then into scripts/
sys.path.insert(0, os.path.join(_notebook_dir, '..', '..', 'scripts'))

import numpy as np
import pandas as pd
from data_utils import REF

PROJECT_ROOT = os.path.abspath(os.path.join(REF, '..'))

# --- paths -------------------------------------------------------------------
DATA_DIR = os.path.join(PROJECT_ROOT, "cleaned_track_data")
IN_PATH  = os.path.join(DATA_DIR, "signatures_model_level.parquet")
OUT_DIR  = os.path.join(_notebook_dir, "outputs")
OUT_PATH = os.path.join(OUT_DIR, "signature_discount.parquet")

# --- column names (fix here if the parquet uses different spellings) ---------
COL_MODEL = "model_id"
COL_MSI = "msiscore"
COL_CIN = "cin"
COL_ANEU = "aneuploidy"

PARAMS = {
    "m_mut": {"m_min": 0.50, "k": 12.0, "p0": 0.93},  # MSI            -> mutations
    "m_fus": {"m_min": 0.70, "k": 5.0, "p0": 0.60},   # CIN+aneuploidy -> fusions
}

print("IN_PATH: ", IN_PATH)
print("OUT_PATH:", OUT_PATH)


IN_PATH:  C:\Disertation\UoB-GeneTraceAI-25-26\cleaned_track_data\signatures_model_level.parquet
OUT_PATH: C:\Disertation\UoB-GeneTraceAI-25-26\src\Track - C\track-C-scoring\outputs\signature_discount.parquet


In [2]:
def floored_reverse_logistic(p, m_min: float, k: float, p0: float):
    """Trust-discount multiplier in [m_min, 1].

    High instability percentile (p -> 1) => m -> m_min (distrust).
    Low  instability percentile (p -> 0) => m -> 1     (safe zone, no penalty).
    NaN percentile propagates to NaN (callers must select 1.0 for those rows).
    """
    return m_min + (1.0 - m_min) * (1.0 / (1.0 + np.exp(k * (p - p0))))


In [3]:
def compute_signature_discount(
    df: pd.DataFrame,
    params: dict = PARAMS,
    *,
    col_model: str = COL_MODEL,
    col_msi: str = COL_MSI,
    col_cin: str = COL_CIN,
    col_aneu: str = COL_ANEU,
) -> pd.DataFrame:
    """Return a per-model_id discount table.

    One row per input row (grain is preserved: model_id -> m_mut, m_fus).
    """
    required = [col_msi, col_cin, col_aneu]
    missing_cols = [c for c in [col_model, *required] if c not in df.columns]
    if missing_cols:
        raise KeyError(
            f"signatures parquet is missing expected columns: {missing_cols}. "
            f"Fix the COL_* constants at the top of this file if the source "
            f"uses different spellings."
        )

    out = df[[col_model]].copy()

    # MSI and the structural (CIN/aneuploidy) metrics have DIFFERENT coverage:
    # MSI is often callable from exome/panel data, while CIN and aneuploidy need
    # broader copy-number/WGS profiling. Gate each discount on ONLY the metric
    # it needs, so an MSI-only line still receives its mutation discount instead
    # of being forced to m = 1.0. (The structural block does drop all-or-nothing;
    # MSI simply survives on more lines than that block does.)
    has_msi = df[col_msi].notna()
    has_struct = df[[col_cin, col_aneu]].notna().all(axis=1)

    # Percentile-rank EACH metric over the non-null cohort (NaN -> NaN, and is
    # excluded from the ranking so it does not distort other lines' percentiles).
    msi_pctl = df[col_msi].rank(pct=True)
    cin_pctl = df[col_cin].rank(pct=True)
    aneu_pctl = df[col_aneu].rank(pct=True)

    # Combined structural instability = mean of the two percentiles, so CIN and
    # aneuploidy contribute equally regardless of their different native scales.
    instab_pctl = pd.concat([cin_pctl, aneu_pctl], axis=1).mean(axis=1)

    out["msi_pctl"] = msi_pctl.to_numpy()
    out["instab_pctl"] = instab_pctl.to_numpy()

    # Apply the discount where the block exists; otherwise NO discount (m = 1.0).
    # We do NOT impute missing instability - penalising a line for absent
    # metadata would fabricate signal (structural nulls are not missing data).
    out["m_mut"] = np.where(
        has_msi,
        floored_reverse_logistic(msi_pctl, **params["m_mut"]),
        1.0,
    )
    out["m_fus"] = np.where(
        has_struct,
        floored_reverse_logistic(instab_pctl, **params["m_fus"]),
        1.0,
    )
    # Two independent audit flags: a line can now legitimately have its mutation
    # discount applied while its fusion discount defaults to 1.0 (or vice versa).
    out["mut_discount_applied"] = has_msi.to_numpy()
    out["fus_discount_applied"] = has_struct.to_numpy()

    return out


In [4]:
def _summary(disc: pd.DataFrame, out_path: str) -> None:
    n = len(disc)
    n_mut = int(disc["mut_discount_applied"].sum())
    n_fus = int(disc["fus_discount_applied"].sum())
    # MSI-only lines: mutation discount available, fusion discount defaulted.
    msi_only = int(
        (disc["mut_discount_applied"] & ~disc["fus_discount_applied"]).sum()
    )
    strong_mut = int((disc["m_mut"] < 0.70).sum())
    strong_fus = int((disc["m_fus"] < 0.85).sum())
    print(f"[signature_discount] wrote {out_path}")
    print(f"  cell lines                       : {n}")
    print(f"  mutation discount available (MSI): {n_mut}  ({n_mut / n:.1%})")
    print(f"  fusion   discount available (CIN): {n_fus}  ({n_fus / n:.1%})")
    print(f"  MSI-only (mut dimmed, fus m=1.0) : {msi_only}   <- recovered by decoupling")
    print(f"  mutations meaningfully dimmed     (m_mut < 0.70): {strong_mut}")
    print(f"  fusions   meaningfully dimmed     (m_fus < 0.85): {strong_fus}")
    print(
        f"  m_mut range                      : "
        f"{disc['m_mut'].min():.3f} - {disc['m_mut'].max():.3f}"
    )
    print(
        f"  m_fus range                      : "
        f"{disc['m_fus'].min():.3f} - {disc['m_fus'].max():.3f}"
    )


## Run

In [5]:
df = pd.read_parquet(IN_PATH)
disc = compute_signature_discount(df)

os.makedirs(OUT_DIR, exist_ok=True)
disc.to_parquet(OUT_PATH, index=False)
_summary(disc, OUT_PATH)


[signature_discount] wrote C:\Disertation\UoB-GeneTraceAI-25-26\src\Track - C\track-C-scoring\outputs\signature_discount.parquet
  cell lines                       : 1955
  mutation discount available (MSI): 1955  (100.0%)
  fusion   discount available (CIN): 1622  (83.0%)
  MSI-only (mut dimmed, fus m=1.0) : 333   <- recovered by decoupling
  mutations meaningfully dimmed     (m_mut < 0.70): 71
  fusions   meaningfully dimmed     (m_fus < 0.85): 697
  m_mut range                      : 0.651 - 1.000
  m_fus range                      : 0.738 - 1.000
